
# Router Pilot 01 — Big-Vul → Static Candidates → Full-5 Cost Measurement (v3)

이 노트북은 **최종 학습용 데이터셋을 만드는 노트북이 아니라, 먼저 API 비용과 Static Candidate Recall을 측정하는 파일럿**입니다.

실행 흐름:

1. 현재 프로젝트 코드를 지정 커밋으로 고정
2. Kaggle Input에서 Big-Vul CSV/Parquet 자동 탐색 + 실제 라벨 분포 검사
3. 취약/정상 함수를 `ProjectCase`로 변환
4. 현재 `SemanticStaticAnalyzer`로 **모든 candidate** 생성
5. 함수 단위 `Static Candidate Recall` 확인
6. candidate 최대 50개 표본 추출
7. 각 candidate에 E1~E5 Full-5 실행 (최대 250 logical calls)
8. 실제 prompt/completion tokens, OpenRouter reported cost, latency 저장
9. 남은 예산으로 가능한 Full-5 candidate 수 계산

> **Kaggle 설정**
>
> - `Add Data`에서 Big-Vul/MSR 데이터셋을 붙이세요. 권장 검색어: `msr-data`, `Big-Vul`.
> - Notebook Settings에서 **Internet = On**
> - Kaggle Secrets에 `OPENROUTER_API_KEY` 추가
> - API 호출 셀은 기본 `RUN_API = False`입니다. Static 분석 결과를 확인한 뒤 직접 `True`로 변경해야 호출됩니다.


In [ ]:

# =========================
# 0. Reproducibility config
# =========================
REPO_URL = "https://github.com/junhyun111/llm-security-feature-web-platform.git"
REPO_COMMIT = "0dc5223b14f9138406cdfd9f999063ebe1f72b8d"

SEED = 2026

# Cost pilot only
PILOT_CANDIDATES = 50
PILOT_VULNERABLE_TARGET = 25
PILOT_SAFE_TARGET = 25

# Candidate 확보용 Big-Vul 함수 row 수.
# candidate가 안 생기는 함수가 있을 수 있어 50개보다 넉넉히 준비합니다.
SOURCE_VULNERABLE_ROWS = 150
SOURCE_SAFE_ROWS = 150
MAX_ROWS_TO_SCAN = 100_000

# OpenRouter pilot
MODEL_ID = "deepseek/deepseek-v4-flash-0731"
MAX_OUTPUT_TOKENS = 2500
MAX_CONCURRENCY = 10

# 5 candidates × 5 experts = 25 calls 단위로 실행 후 예산 확인
API_BATCH_CANDIDATES = 5
MAX_API_CALLS = PILOT_CANDIDATES * 5

# 파일럿 중 실수로 잔액을 크게 쓰지 않도록 강제 중단.
# usage.cost가 OpenRouter 응답에 포함된 경우 이 한도를 사용합니다.
MAX_PILOT_COST_USD = 1.00

# 반드시 Static 분석 결과를 본 다음 True로 변경
RUN_API = False

# 이후 전체 실험용으로 남겨둘 목표 예산
PLANNED_EXPERIMENT_BUDGET_USD = 20.0
RESERVE_BUDGET_USD = 5.0

print({
    "repo_commit": REPO_COMMIT,
    "model": MODEL_ID,
    "pilot_candidates": PILOT_CANDIDATES,
    "max_logical_calls": MAX_API_CALLS,
    "pilot_cost_guard_usd": MAX_PILOT_COST_USD,
    "run_api": RUN_API,
})


## 1. 프로젝트 코드와 최소 의존성 준비

In [ ]:

# Kaggle image에 이미 있는 큰 ML 패키지를 다시 설치하지 않고,
# 현재 Static Analyzer + OpenRouter 실행에 필요한 최소 패키지만 설치합니다.
%pip install -q "httpx>=0.27,<1" "tree-sitter>=0.25,<0.26" "tree-sitter-c>=0.24,<0.25" "tree-sitter-cpp>=0.23,<0.24"


In [ ]:

from pathlib import Path
import ast
import os, sys, subprocess, random, json, hashlib, math, re, time
import pandas as pd
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

WORK_ROOT = Path("/kaggle/working/router_pilot")
REPO_DIR = WORK_ROOT / "repo"
OUT_DIR = WORK_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_COMMIT], check=True)

SRC_DIR = REPO_DIR / "model_runtime" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert head == REPO_COMMIT, (head, REPO_COMMIT)

print("Repository:", REPO_DIR)
print("Pinned commit:", head)
print("Output dir:", OUT_DIR)



## 2. Big-Vul 데이터 파일을 **라벨 열부터 먼저** 검사

v3의 핵심 변경점입니다.

기존 버전은 CSV 앞부분의 3,000개 row만 보고 취약 row 존재 여부를 추정했습니다.
Big-Vul처럼 정상 함수가 압도적으로 많거나 데이터가 정렬된 경우 이 방식은 `vul=1`을 놓칠 수 있습니다.

이번 버전은:

1. `/kaggle/input`에서 Big-Vul/MSR 관련 디렉터리를 우선 탐색
2. 각 CSV/Parquet의 **라벨 열만** 먼저 chunk streaming
3. `0/1` 실제 개수를 확인
4. `1`이 존재하는 파일을 최우선으로 선택
5. 그 뒤에만 함수 코드 열을 읽습니다.


In [ ]:

INPUT_ROOT = Path("/kaggle/input")

# 라벨 탐색은 코드 본문을 읽지 않으므로 비교적 가볍습니다.
LABEL_SCAN_CHUNK = 100_000

# None이면 CSV 라벨 열을 끝까지 스캔합니다.
# 너무 큰 파생 dataset을 붙인 경우에만 숫자를 지정하세요.
MAX_LABEL_ROWS_PER_FILE = None

def normalized_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")

def find_column(columns, *aliases):
    by_norm = {normalized_name(c): c for c in columns}
    for alias in aliases:
        hit = by_norm.get(normalized_name(alias))
        if hit is not None:
            return hit
    return None

def normalize_vul(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, (bool, np.bool_)):
        return int(value)

    text = str(value).strip().lower()
    positive = {
        "1", "1.0", "true", "yes", "y",
        "vulnerable", "vul", "positive", "pos"
    }
    negative = {
        "0", "0.0", "false", "no", "n",
        "safe", "non-vulnerable", "non_vulnerable",
        "nonvulnerable", "negative", "neg"
    }
    if text in positive:
        return 1
    if text in negative:
        return 0

    try:
        number = float(text)
        if number == 1:
            return 1
        if number == 0:
            return 0
    except Exception:
        pass
    return None

def valid_code(value):
    if not isinstance(value, str):
        return False
    text = value.strip()
    return len(text) >= 20 and "{" in text and "}" in text

def table_columns(path: Path):
    try:
        suffix = path.suffix.lower()
        if suffix == ".csv":
            return list(pd.read_csv(path, nrows=0).columns)
        if suffix in {".parquet", ".pq"}:
            import pyarrow.parquet as pq
            return pq.ParquetFile(path).schema.names
    except Exception as e:
        print("HEADER SKIP:", path, type(e).__name__, e)
    return []

def source_priority(path: Path) -> int:
    text = str(path).lower()
    score = 0
    # 사용자가 권장 Kaggle dataset을 Add Data로 붙였을 때 보통 msr-data 디렉터리가 됩니다.
    if "/msr-data/" in text or text.endswith("/msr-data"):
        score += 100
    if "big-vul" in text or "big_vul" in text or "bigvul" in text:
        score += 80
    if "msr" in text:
        score += 30
    if "clean" in text:
        score += 10
    return score

def scan_label_distribution(path: Path, label_col: str):
    pos = neg = unknown = rows = 0
    raw_values = set()

    try:
        if path.suffix.lower() == ".csv":
            iterator = pd.read_csv(
                path,
                usecols=[label_col],
                chunksize=LABEL_SCAN_CHUNK,
                low_memory=False,
            )
        else:
            df = pd.read_parquet(path, columns=[label_col])
            iterator = [df]

        for chunk in iterator:
            series = chunk[label_col]
            if len(raw_values) < 20:
                for value in series.dropna().astype(str).unique()[:20]:
                    raw_values.add(value[:100])

            labels = series.map(normalize_vul)
            pos += int((labels == 1).sum())
            neg += int((labels == 0).sum())
            unknown += int(labels.isna().sum())
            rows += len(chunk)

            if MAX_LABEL_ROWS_PER_FILE is not None and rows >= MAX_LABEL_ROWS_PER_FILE:
                break

    except Exception as e:
        return {
            "positive": 0,
            "negative": 0,
            "unknown": 0,
            "rows_scanned": rows,
            "raw_values": sorted(raw_values),
            "scan_error": f"{type(e).__name__}: {e}",
        }

    return {
        "positive": pos,
        "negative": neg,
        "unknown": unknown,
        "rows_scanned": rows,
        "raw_values": sorted(raw_values),
        "scan_error": "",
    }

table_files = (
    list(INPUT_ROOT.rglob("*.csv"))
    + list(INPUT_ROOT.rglob("*.parquet"))
    + list(INPUT_ROOT.rglob("*.pq"))
)

print("Total CSV/Parquet files under /kaggle/input:", len(table_files))

dataset_files = []

for file_no, path in enumerate(sorted(table_files, key=lambda p: (-source_priority(p), str(p))), start=1):
    cols = table_columns(path)
    if not cols:
        continue

    func_col = find_column(
        cols,
        "func_before", "function_before", "before",
        "func", "code", "function"
    )
    label_col = find_column(
        cols,
        "vul", "target", "label", "is_vulnerable",
        "vulnerable", "y"
    )
    after_col = find_column(
        cols,
        "func_after", "function_after", "after", "fixed_func"
    )

    if func_col is None or label_col is None:
        continue

    print(f"[{file_no}] label scan:", path)
    distribution = scan_label_distribution(path, label_col)

    dataset_files.append({
        "path": path,
        "columns": cols,
        "func_col": func_col,
        "label_col": label_col,
        "after_col": after_col,
        "source_priority": source_priority(path),
        **distribution,
    })

if not dataset_files:
    raise FileNotFoundError(
        "함수 코드 열(func_before/func/code)과 라벨 열(vul/target/label)을 함께 가진 "
        "CSV/Parquet을 찾지 못했습니다.\n"
        "Kaggle Add Data에서 정확히 `Big-Vul` (owner: kaggler10240, slug: msr-data)을 "
        "추가하는 것을 권장합니다."
    )

# 실제 positive 존재 여부를 최우선으로 정렬
dataset_files.sort(
    key=lambda x: (
        -(x["positive"] > 0),
        -(x["positive"] > 0 and x["negative"] > 0),
        -x["source_priority"],
        -x["positive"],
        -x["negative"],
        str(x["path"]),
    )
)

diagnostic = pd.DataFrame([
    {
        "file": str(info["path"]),
        "func_col": info["func_col"],
        "label_col": info["label_col"],
        "after_col": info["after_col"],
        "rows_scanned": info["rows_scanned"],
        "positive": info["positive"],
        "negative": info["negative"],
        "unknown": info["unknown"],
        "raw_label_values": " | ".join(info["raw_values"][:10]),
        "scan_error": info["scan_error"],
        "source_priority": info["source_priority"],
    }
    for info in dataset_files
])

pd.set_option("display.max_colwidth", 120)
display(diagnostic)

positive_sources = [x for x in dataset_files if x["positive"] > 0]
negative_sources = [x for x in dataset_files if x["negative"] > 0]

print("\nFiles containing vulnerable rows:", len(positive_sources))
print("Files containing safe rows:", len(negative_sources))

if not positive_sources:
    print("\n=== IMPORTANT ===")
    print(
        "현재 /kaggle/input 안에서 함수 코드 + 라벨을 가진 파일을 끝까지/설정 한도까지 "
        "스캔했지만 vulnerable label을 찾지 못했습니다."
    )
    print(
        "이 경우 임의로 취약 라벨을 만들어서는 안 됩니다. "
        "Kaggle에서 `kaggler10240/msr-data` Big-Vul 데이터셋을 Add Data로 붙인 뒤 재실행하세요."
    )
    raise RuntimeError(
        "Big-Vul vulnerable rows not found. "
        "See the diagnostic table above and attach Kaggle dataset `kaggler10240/msr-data`."
    )

BIGVUL_SOURCES = dataset_files
print("\nPrimary vulnerable source:", positive_sources[0]["path"])



## 3. 확인된 positive 파일에서 실제 함수 코드 표본 수집

이 셀은 앞 단계에서 **`positive > 0`이 확인된 파일을 먼저 읽습니다.**

중요하게도 더 이상 파일 전체에 공유되는 `100,000 row` 한도가 없습니다.
각 파일을 독립적으로 streaming하며, 원하는 취약/정상 함수 수를 확보하면 즉시 종료합니다.


In [ ]:

def safe_scalar(value, default=""):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    return str(value).strip()

def cwe_column(columns):
    return find_column(columns, "cwe_id", "CWE ID", "cwe", "cwe_ids")

def project_column(columns):
    return find_column(columns, "project", "project_name", "repo", "repository")

def cve_column(columns):
    return find_column(columns, "cve_id", "CVE ID", "cve")

def commit_column(columns):
    return find_column(columns, "commit_id", "commit", "commit_hash")

def lang_column(columns):
    return find_column(columns, "lang", "language")

def lines_before_column(columns):
    return find_column(
        columns,
        "lines_before",
        "line_before",
        "changed_lines_before",
    )

def parse_flaw_lines(value):
    if not value:
        return []
    if isinstance(value, (list, tuple, set)):
        return sorted({int(x) for x in value if str(x).strip().isdigit()})

    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return sorted({
                int(x) for x in parsed if str(x).strip().isdigit()
            })
    except (SyntaxError, ValueError):
        pass

    return sorted({int(x) for x in re.findall(r"\d+", text)})

def normalized_record(row, info, *, code, label, code_source, row_index):
    cols = info["columns"]
    return {
        "code": str(code).strip(),
        "label": int(label),
        "project": safe_scalar(row.get(project_column(cols)), "unknown-project"),
        "cve_id": safe_scalar(row.get(cve_column(cols)), ""),
        "commit_id": safe_scalar(row.get(commit_column(cols)), ""),
        "cwe_raw": safe_scalar(row.get(cwe_column(cols)), ""),
        "lang": safe_scalar(row.get(lang_column(cols)), ""),
        "lines_before_raw": safe_scalar(
            row.get(lines_before_column(cols)), ""
        ),
        "source_path": str(info["path"]),
        "code_source": code_source,
        "source_row_index": int(row_index) if isinstance(row_index, (int, np.integer)) else str(row_index),
    }

def columns_needed(info):
    wanted = [
        info["func_col"],
        info["label_col"],
        info["after_col"],
        cwe_column(info["columns"]),
        project_column(info["columns"]),
        cve_column(info["columns"]),
        commit_column(info["columns"]),
        lang_column(info["columns"]),
        lines_before_column(info["columns"]),
    ]
    return list(dict.fromkeys(x for x in wanted if x))

def iter_source_chunks(info):
    path = info["path"]
    usecols = columns_needed(info)

    if path.suffix.lower() == ".csv":
        yield from pd.read_csv(
            path,
            usecols=usecols,
            chunksize=5000,
            low_memory=False,
        )
    else:
        yield pd.read_parquet(path, columns=usecols)

vuln_records = []
safe_records = []
fixed_safe_pool = []

seen_vuln = set()
seen_safe = set()

# positive가 실제 확인된 파일을 가장 먼저 읽고,
# 그 다음 mixed/negative 파일을 사용
ordered_sources = sorted(
    BIGVUL_SOURCES,
    key=lambda x: (
        -(x["positive"] > 0),
        -(x["positive"] > 0 and x["negative"] > 0),
        -x["positive"],
        -x["negative"],
        -x["source_priority"],
    )
)

for info in ordered_sources:
    need_vuln = len(vuln_records) < SOURCE_VULNERABLE_ROWS
    need_safe = len(safe_records) < SOURCE_SAFE_ROWS

    if not need_vuln and not need_safe:
        break

    # 이 파일에 필요한 class가 하나도 없으면 skip
    if (not need_vuln or info["positive"] == 0) and (not need_safe or info["negative"] == 0):
        continue

    print(
        f"\nCollecting from: {info['path']}\n"
        f"  labels: positive={info['positive']}, negative={info['negative']}\n"
        f"  columns: func={info['func_col']}, label={info['label_col']}, after={info['after_col']}"
    )

    local_rows = 0

    for chunk in iter_source_chunks(info):
        # 파일 내 row 순서를 일부 섞되 chunk 자체는 순차 streaming
        chunk = chunk.sample(
            frac=1.0,
            random_state=SEED + (local_rows % 1009)
        )

        for row_index, row in chunk.iterrows():
            local_rows += 1

            before = row.get(info["func_col"])
            label = normalize_vul(row.get(info["label_col"]))

            if label is None or not valid_code(before):
                continue

            before_text = str(before).strip()
            before_hash = hashlib.sha1(
                before_text.encode("utf-8", errors="ignore")
            ).hexdigest()

            if (
                label == 1
                and len(vuln_records) < SOURCE_VULNERABLE_ROWS
                and before_hash not in seen_vuln
            ):
                vuln_records.append(
                    normalized_record(
                        row,
                        info,
                        code=before_text,
                        label=1,
                        code_source="func_before",
                        row_index=row_index,
                    )
                )
                seen_vuln.add(before_hash)

                # 같은 취약 함수의 수정 후 버전은 safe fallback 후보로 보관
                if info["after_col"]:
                    after = row.get(info["after_col"])
                    if valid_code(after):
                        after_text = str(after).strip()
                        after_hash = hashlib.sha1(
                            after_text.encode("utf-8", errors="ignore")
                        ).hexdigest()
                        if after_text != before_text and after_hash not in seen_safe:
                            fixed_safe_pool.append(
                                normalized_record(
                                    row,
                                    info,
                                    code=after_text,
                                    label=0,
                                    code_source="func_after_fixed_fallback",
                                    row_index=row_index,
                                )
                            )
                            seen_safe.add(after_hash)

            elif (
                label == 0
                and len(safe_records) < SOURCE_SAFE_ROWS
                and before_hash not in seen_safe
            ):
                safe_records.append(
                    normalized_record(
                        row,
                        info,
                        code=before_text,
                        label=0,
                        code_source="func_before",
                        row_index=row_index,
                    )
                )
                seen_safe.add(before_hash)

            if (
                len(vuln_records) >= SOURCE_VULNERABLE_ROWS
                and len(safe_records) >= SOURCE_SAFE_ROWS
            ):
                break

        print(
            f"  progress after {local_rows:,} rows: "
            f"vulnerable={len(vuln_records)}, safe={len(safe_records)}"
        )

        if (
            len(vuln_records) >= SOURCE_VULNERABLE_ROWS
            and len(safe_records) >= SOURCE_SAFE_ROWS
        ):
            break

# safe가 부족한 파생 Big-Vul 파일용 fallback
if len(safe_records) < SOURCE_SAFE_ROWS and fixed_safe_pool:
    needed = SOURCE_SAFE_ROWS - len(safe_records)
    random.Random(SEED).shuffle(fixed_safe_pool)
    safe_records.extend(fixed_safe_pool[:needed])

print("\n========== COLLECTION SUMMARY ==========")
print("Vulnerable source rows:", len(vuln_records))
print("Safe source rows:", len(safe_records))
print(
    "Safe rows from fixed func_after:",
    sum(r["code_source"] == "func_after_fixed_fallback" for r in safe_records),
)

if not vuln_records:
    display(diagnostic)
    raise RuntimeError(
        "라벨 스캔에서는 positive row가 있었지만 유효한 C/C++ func_before를 확보하지 못했습니다. "
        "diagnostic의 func_col과 실제 데이터 형식을 확인해야 합니다."
    )

if not safe_records:
    print(
        "WARNING: safe row는 확보하지 못했습니다. "
        "비용 측정 pilot은 계속 가능하지만 FPR 관련 수치는 사용하지 마세요."
    )


## 4. Big-Vul row → `ProjectCase` 변환

In [ ]:

from llm_security.models import ProjectCase, GroundTruth, to_dict
from llm_security.cwe import expert_for_cwe

def parse_cwes(value):
    if value is None:
        return []
    found = re.findall(r"CWE[-_ ]?(\d+)", str(value), flags=re.I)
    return sorted({f"CWE-{x}" for x in found})

def make_case(record, index):
    code = record["code"]
    label = int(record["label"])
    project = record["project"] or "unknown-project"
    cve = record["cve_id"]
    commit = record["commit_id"]
    lang = record["lang"].lower()

    ext = ".cpp" if ("++" in lang or "cpp" in lang or "cxx" in lang) else ".c"
    digest = hashlib.sha1(
        (
            f"{project}|{cve}|{commit}|{record['source_path']}|"
            f"{record['source_row_index']}|{record['code_source']}|{code[:200]}"
        ).encode("utf-8", errors="ignore")
    ).hexdigest()[:12]

    filename = f"bigvul_{digest}{ext}"
    nlines = max(1, len(code.splitlines()))
    cwes = parse_cwes(record["cwe_raw"])

    experts = []
    for cwe in cwes:
        try:
            expert = expert_for_cwe(cwe)
        except Exception:
            expert = None
        if expert is not None and expert not in experts:
            experts.append(expert)

    truths = []
    if label == 1:
        truths = [
            GroundTruth(
                truth_id=f"T-{digest}",
                file=filename,
                # pilot에서는 함수 단위 truth입니다.
                # 정확한 함수명/line-level GT는 최종 데이터셋 준비 notebook에서 보강합니다.
                # Empty function is a RecallTracer function-name wildcard.
                function="",
                line_start=1,
                line_end=nlines,
                experts=experts,
                cwes=cwes,
            )
        ]

    return ProjectCase(
        case_id=f"BV-{digest}",
        project_id=project,
        source_files={filename: code},
        split="pilot",
        vulnerable_revision=commit or None,
        ground_truth=truths,
        metadata={
            "dataset": "Big-Vul",
            "label": label,
            "cve_id": cve,
            "commit_id": commit,
            "cwes": cwes,
            "flaw_lines": parse_flaw_lines(record["lines_before_raw"]),
            "source_path": record["source_path"],
            "source_row_index": record["source_row_index"],
            "code_source": record["code_source"],
            "fixed_safe_fallback": (
                record["code_source"] == "func_after_fixed_fallback"
            ),
        },
    )

records = (
    [(1, r) for r in vuln_records]
    + [(0, r) for r in safe_records]
)
random.Random(SEED).shuffle(records)

cases = [make_case(record, i) for i, (_, record) in enumerate(records)]

cases_path = OUT_DIR / "pilot_cases.jsonl"
with cases_path.open("w", encoding="utf-8") as f:
    for case in cases:
        f.write(json.dumps(to_dict(case), ensure_ascii=False) + "\n")

case_summary = pd.DataFrame([
    {
        "case_id": c.case_id,
        "label": c.metadata["label"],
        "project": c.project_id,
        "cve": c.metadata["cve_id"],
        "code_source": c.metadata["code_source"],
        "cwes": ",".join(c.metadata["cwes"]),
    }
    for c in cases
])

print("Cases:", len(cases))
print("Saved:", cases_path)
display(
    case_summary.groupby(["label", "code_source"])
    .size()
    .rename("count")
    .reset_index()
)


## 5. 현재 Semantic Static Analyzer로 모든 candidate 생성

In [ ]:

from llm_security.analysis import SemanticStaticAnalyzer

analyzer = SemanticStaticAnalyzer(
    max_source_bytes=2 * 1024 * 1024,
    parse_timeout_ms=30_000,
)

candidate_records = []
case_errors = []

for i, case in enumerate(cases, start=1):
    try:
        generated = analyzer.analyze(case)
    except Exception as e:
        case_errors.append({
            "case_id": case.case_id,
            "project_id": case.project_id,
            "error": f"{type(e).__name__}: {e}",
        })
        continue

    label = int(case.metadata.get("label", 0))
    for cand in generated:
        candidate_records.append({
            "case_id": case.case_id,
            "project_id": case.project_id,
            "label": label,
            "cve_id": case.metadata.get("cve_id", ""),
            "ground_truth_cwes": case.metadata.get("cwes", []),
            "candidate": cand,
        })

    if i % 50 == 0:
        print(f"{i}/{len(cases)} cases analyzed, candidates={len(candidate_records)}")

print("Generated candidates:", len(candidate_records))
print("Static analysis errors:", len(case_errors))

errors_path = OUT_DIR / "static_analysis_errors.jsonl"
with errors_path.open("w", encoding="utf-8") as f:
    for row in case_errors:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


## 6. 함수 단위 Static Candidate Recall 측정

In [ ]:

vulnerable_cases = [
    case for case in cases if int(case.metadata.get("label", 0)) == 1
]
vulnerable_case_ids = {case.case_id for case in vulnerable_cases}
safe_case_ids = {
    case.case_id for case in cases if int(case.metadata.get("label", 0)) == 0
}
candidate_case_ids = {row["case_id"] for row in candidate_records}

def candidate_covers_flaw(candidate, flaw_lines):
    return any(
        candidate.line_start <= line <= candidate.line_end
        for line in flaw_lines
    )

vulnerable_cases_with_flaw_lines = [
    case for case in vulnerable_cases if case.metadata.get("flaw_lines")
]
covered_vulnerable = [
    case
    for case in vulnerable_cases_with_flaw_lines
    if any(
        candidate_covers_flaw(row["candidate"], case.metadata["flaw_lines"])
        for row in candidate_records
        if row["case_id"] == case.case_id
    )
]
static_candidate_recall = (
    len(covered_vulnerable) / len(vulnerable_cases_with_flaw_lines)
    if vulnerable_cases_with_flaw_lines else float("nan")
)

summary_static = {
    "vulnerable_cases": len(vulnerable_case_ids),
    "vulnerable_cases_with_flaw_lines": len(vulnerable_cases_with_flaw_lines),
    "vulnerable_cases_with_flaw_covering_candidate": len(covered_vulnerable),
    "flaw_line_static_candidate_recall": static_candidate_recall,
    "safe_cases": len(safe_case_ids),
    "safe_cases_with_candidate": len(safe_case_ids & candidate_case_ids),
    "total_candidates": len(candidate_records),
    "analysis_errors": len(case_errors),
}

display(pd.DataFrame([summary_static]))

# candidate 데이터도 저장
candidate_path = OUT_DIR / "all_pilot_candidates.jsonl"
with candidate_path.open("w", encoding="utf-8") as f:
    for row in candidate_records:
        serializable = dict(row)
        serializable["candidate"] = to_dict(row["candidate"])
        f.write(json.dumps(serializable, ensure_ascii=False) + "\n")

print("Saved:", candidate_path)

# 중요 경고
if vulnerable_case_ids and static_candidate_recall < 0.80:
    print(
        "\nWARNING: Static Candidate Recall < 0.80. "
        "대규모 API 호출 전에 Static Analyzer 병목을 먼저 확인하는 것이 좋습니다."
    )


## 7. Full-5 비용 측정용 candidate 50개 표본

In [ ]:

rng = random.Random(SEED)

vul_candidates = [r for r in candidate_records if r["label"] == 1]
safe_candidates = [r for r in candidate_records if r["label"] == 0]

rng.shuffle(vul_candidates)
rng.shuffle(safe_candidates)

pilot_records = (
    vul_candidates[:PILOT_VULNERABLE_TARGET]
    + safe_candidates[:PILOT_SAFE_TARGET]
)

# 한쪽 class가 부족하면 나머지 candidate로 50개까지 채움
selected_ids = {r["candidate"].candidate_id for r in pilot_records}
remaining = [
    r for r in candidate_records
    if r["candidate"].candidate_id not in selected_ids
]
rng.shuffle(remaining)
pilot_records.extend(remaining[: max(0, PILOT_CANDIDATES - len(pilot_records))])
pilot_records = pilot_records[:PILOT_CANDIDATES]

if len(pilot_records) < PILOT_CANDIDATES:
    print(
        f"WARNING: candidate가 {len(pilot_records)}개뿐입니다. "
        "SOURCE_*_ROWS를 늘린 뒤 Static 분석 셀부터 다시 실행하세요."
    )

pilot_candidates = [r["candidate"] for r in pilot_records]

pilot_index = pd.DataFrame([
    {
        "candidate_id": r["candidate"].candidate_id,
        "case_id": r["case_id"],
        "project_id": r["project_id"],
        "label": r["label"],
        "cve_id": r["cve_id"],
        "gt_cwes": ",".join(r["ground_truth_cwes"]),
        "file": r["candidate"].file,
        "function": r["candidate"].function,
        "line_start": r["candidate"].line_start,
        "line_end": r["candidate"].line_end,
        "suspicion_score": r["candidate"].suspicion_score,
        "static_cwe_hypotheses": ",".join(h.cwe for h in r["candidate"].cwe_hypotheses),
    }
    for r in pilot_records
])

display(pilot_index.head(20))
print("\nPilot candidates:", len(pilot_candidates))
print(pilot_index["label"].value_counts(dropna=False).rename({0:"safe",1:"vulnerable"}))

pilot_index.to_csv(OUT_DIR / "pilot_candidate_index.csv", index=False)



## 8. OpenRouter Secret 로드

Kaggle → **Add-ons / Secrets**에서 `OPENROUTER_API_KEY`를 등록하세요.

이 셀은 key를 출력하지 않습니다.


In [ ]:

if RUN_API:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Kaggle Secret OPENROUTER_API_KEY가 없습니다.")
    print("OPENROUTER_API_KEY loaded:", bool(OPENROUTER_API_KEY))
else:
    OPENROUTER_API_KEY = None
    print("RUN_API=False — 아직 API를 호출하지 않습니다.")



## 9. Full-5 Pilot 실행

**실행 전 확인**

- 위 Static Candidate Recall이 정상적인지 확인
- `PILOT_CANDIDATES <= 50`인지 확인
- `MAX_API_CALLS <= 250`인지 확인
- `MAX_PILOT_COST_USD`가 원하는 안전 한도인지 확인
- 첫 설정 셀에서 `RUN_API = True`로 바꾼 후 이 셀 실행

5 candidate씩 묶어서 E1~E5를 병렬 실행하고, 매 batch마다 OpenRouter가 돌려준 `usage.cost`를 합산합니다.


In [ ]:

from llm_security.models import ACTIVE_UTILITY_EXPERTS
from llm_security.llm import OpenRouterClient
from llm_security.evidence import ContextBuilder
from llm_security.experts import ParallelExpertRunner

if RUN_API:
    if len(pilot_candidates) > PILOT_CANDIDATES:
        raise RuntimeError("Pilot candidate cap exceeded")
    if len(pilot_candidates) * len(ACTIVE_UTILITY_EXPERTS) > MAX_API_CALLS:
        raise RuntimeError("API call cap exceeded")

    client = OpenRouterClient(
        api_key=OPENROUTER_API_KEY,
        timeout_seconds=90.0,
        max_retries=1,
        temperature=0.0,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        reasoning_enabled=False,
        provider_sort="throughput",
        provider_ignore=("baidu",),
        require_parameters=True,
        allow_fallbacks=True,
        structured_output=True,
        json_repair=True,
        structured_output_fallback=True,
    )

    runner = ParallelExpertRunner(
        client=client,
        model=MODEL_ID,
        context_builder=ContextBuilder(max_characters=30_000),
        max_concurrency=MAX_CONCURRENCY,
        recovery_attempts=1,
    )

    raw_results = []
    raw_failures = []
    running_cost = 0.0
    submitted_calls = 0

    candidate_meta = {
        r["candidate"].candidate_id: r
        for r in pilot_records
    }

    for batch_start in range(0, len(pilot_candidates), API_BATCH_CANDIDATES):
        if submitted_calls >= MAX_API_CALLS:
            print("STOP: MAX_API_CALLS reached")
            break
        if running_cost >= MAX_PILOT_COST_USD:
            print("STOP: MAX_PILOT_COST_USD reached")
            break

        batch = pilot_candidates[
            batch_start : batch_start + API_BATCH_CANDIDATES
        ]
        experts_by_candidate = {
            c.candidate_id: list(ACTIVE_UTILITY_EXPERTS)
            for c in batch
        }

        expected_calls = sum(len(v) for v in experts_by_candidate.values())
        if submitted_calls + expected_calls > MAX_API_CALLS:
            print("STOP: next batch would exceed MAX_API_CALLS")
            break

        print(
            f"\nBatch {batch_start // API_BATCH_CANDIDATES + 1}: "
            f"{len(batch)} candidates / {expected_calls} logical calls"
        )

        output = runner.run_experts(
            batch,
            experts_by_candidate,
            phase="pilot_full5",
        )
        submitted_calls += output.submitted_task_count

        # ParallelExpertRunner는 성공한 task에 대해 assessments와 usage를
        # 동일 task 순서로 정렬하여 반환합니다.
        for assessment, usage in zip(output.assessments, output.usage):
            meta = candidate_meta[assessment.candidate_id]
            row = {
                "case_id": meta["case_id"],
                "project_id": meta["project_id"],
                "dataset_label": meta["label"],
                "cve_id": meta["cve_id"],
                "ground_truth_cwes": meta["ground_truth_cwes"],
                "candidate_id": assessment.candidate_id,
                "expert": assessment.expert.value,
                "verdict": assessment.verdict.value,
                "reported_cwes": assessment.cwes,
                "evidence_ids": assessment.evidence_ids,
                "counter_evidence_ids": assessment.counter_evidence_ids,
                "confidence": assessment.confidence,
                "model_id": assessment.model_id,
                "prompt_version": assessment.prompt_version,
                "prompt_tokens": usage.prompt_tokens,
                "completion_tokens": usage.completion_tokens,
                "reasoning_tokens": usage.reasoning_tokens,
                "cost_usd": usage.cost,
                "latency_seconds": usage.latency_seconds,
                "provider": usage.provider,
            }
            raw_results.append(row)
            running_cost += float(usage.cost or 0.0)

        for failure in output.failures:
            raw_failures.append(to_dict(failure))

        print(
            f" completed={output.completed_task_count}, "
            f"failed={output.failed_task_count}, "
            f"cumulative_calls={submitted_calls}, "
            f"reported_cost=${running_cost:.6f}"
        )

        if running_cost >= MAX_PILOT_COST_USD:
            print(
                f"STOP: reported cumulative cost ${running_cost:.6f} "
                f">= guard ${MAX_PILOT_COST_USD:.2f}"
            )
            break

        # 매 batch마다 checkpoint 저장
        with (OUT_DIR / "pilot_full5_raw.jsonl").open("w", encoding="utf-8") as f:
            for row in raw_results:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        with (OUT_DIR / "pilot_full5_failures.jsonl").open("w", encoding="utf-8") as f:
            for row in raw_failures:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("\nPilot API phase complete.")
else:
    print(
        "RUN_API=False입니다. Static 결과를 먼저 검토한 뒤 "
        "첫 설정 셀의 RUN_API=True로 변경하세요."
    )


## 10. 실제 비용/토큰 통계와 $20 실험 예산 산정

In [ ]:

results_path = OUT_DIR / "pilot_full5_raw.jsonl"

if results_path.exists():
    pilot_api = pd.read_json(results_path, lines=True)

    if len(pilot_api):
        metrics = {
            "successful_calls": len(pilot_api),
            "unique_candidates": pilot_api["candidate_id"].nunique(),
            "total_prompt_tokens": int(pilot_api["prompt_tokens"].sum()),
            "total_completion_tokens": int(pilot_api["completion_tokens"].sum()),
            "total_reported_cost_usd": float(pilot_api["cost_usd"].sum()),
            "mean_prompt_tokens": float(pilot_api["prompt_tokens"].mean()),
            "mean_completion_tokens": float(pilot_api["completion_tokens"].mean()),
            "mean_cost_per_call_usd": float(pilot_api["cost_usd"].mean()),
            "p95_cost_per_call_usd": float(pilot_api["cost_usd"].quantile(0.95)),
            "mean_latency_seconds": float(pilot_api["latency_seconds"].mean()),
            "p95_latency_seconds": float(pilot_api["latency_seconds"].quantile(0.95)),
        }
        display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

        mean_cost = metrics["mean_cost_per_call_usd"]
        p95_cost = metrics["p95_cost_per_call_usd"]

        budget_rows = []
        for label, unit_cost in [
            ("mean-cost estimate", mean_cost),
            ("p95-cost conservative", p95_cost),
        ]:
            if unit_cost > 0:
                affordable_calls = int(PLANNED_EXPERIMENT_BUDGET_USD // unit_cost)
                affordable_full5_candidates = affordable_calls // 5
                budget_rows.append({
                    "basis": label,
                    "budget_usd": PLANNED_EXPERIMENT_BUDGET_USD,
                    "unit_cost_usd": unit_cost,
                    "affordable_expert_calls": affordable_calls,
                    "affordable_full5_candidates": affordable_full5_candidates,
                })

        if budget_rows:
            print("\n$20 experiment budget capacity:")
            display(pd.DataFrame(budget_rows))
        else:
            print(
                "OpenRouter usage.cost가 0으로 반환되었습니다. "
                "OpenRouter dashboard의 실제 spend와 token 통계를 함께 확인하세요."
            )

        # Expert / provider별 분포
        print("\nPer-expert:")
        display(
            pilot_api.groupby("expert")
            .agg(
                calls=("expert", "size"),
                avg_prompt=("prompt_tokens", "mean"),
                avg_completion=("completion_tokens", "mean"),
                avg_cost=("cost_usd", "mean"),
                avg_latency=("latency_seconds", "mean"),
            )
            .sort_index()
        )

        print("\nProviders:")
        display(
            pilot_api.groupby("provider", dropna=False)
            .agg(
                calls=("provider", "size"),
                cost=("cost_usd", "sum"),
                avg_latency=("latency_seconds", "mean"),
            )
            .sort_values("calls", ascending=False)
        )
    else:
        print("pilot_full5_raw.jsonl은 있으나 성공 결과가 없습니다.")
else:
    print("아직 API pilot 결과가 없습니다.")


## 11. 산출물 묶기

In [ ]:

import shutil

archive_base = Path("/kaggle/working/router_pilot_results")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(OUT_DIR),
)

print("Saved output files:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size, "bytes")

print("\nArchive:", archive_path)



## 파일럿이 끝나면 확인할 숫자

다음 6개 숫자만 가져오면 다음 실험 규모를 계산할 수 있습니다.

- `function_level_static_candidate_recall`
- `mean_prompt_tokens`
- `mean_completion_tokens`
- `mean_cost_per_call_usd`
- `p95_cost_per_call_usd`
- 성공/실패 API call 수

그 결과를 기준으로 다음 단계에서 **Big-Vul project-disjoint train/dev/test split과 Router Full-5 outcome 수집 규모**를 `$25 잔액 안에서 확정합니다.
